# PINNs for 2D Allen-Cahn Equation with R3 Adaptive Sampling and Causal Weighting

This notebook implements a Physics-Informed Neural Network (PINN) for the **2D Allen–Cahn equation**

$$
    \partial_t u = \varepsilon^2 \Delta u - (u^3 - u)
$$

on the domain $(t, x,y) \in [0,T_f] \times [0,1]^2$ with periodic boundary conditions using JAX, Equinox, Optax, R3 (Retain-Resample-Release) dynamic sampling, and causality training.


In this notebook, we will work on: 
- Network initialization with Jax and Equinox
- Sampling & visualization of training points
- Training with **adaptive sampling** and **causality training**
- Solution visualization on a grid
- First step to Fourier Neural Operator (FNO)

In [ ]:
!pip install "jax[cuda12]" matplotlib numpy scipy optax equinox jaxtyping tqdm --no-cache-dir

In [ ]:

from typing import Tuple, List, Callable, Dict, Any, NamedTuple, Optional
from functools import partial
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import optax
from scipy.stats import qmc
from tqdm import trange
from utils.helper import *

# Enforce 64-bit precision for physical PDE resolution if required
jax.config.update("jax_enable_x64", True)
# Print environment info
jax.print_environment_info()


### Physics & Domain Hyperparameters
First, we set up domain bounds, grid resolution, and physical parameters:

In [ ]:

config = Config(
    x_lb = 0.0,
    x_rb = 1.0,
    y_lb = 0.0,
    y_ub = 1.0,
    eps = 2.5e-2,  # Interface width parameter
    T0 = 0.0,
    Tf = 16.0,  # Reduced Tf for stable baseline execution
    N_t = 32, # Number of temporal slices
    N_x = 160,  # Spatial resolution x
    N_y = 160,  # Spatial resolution y
    n_PDE = 200, # Spatial points per time slice
    n_IC = 200,  # Initial condition collocation points
    n_BC = 100,  # Boundary condition collocation points
    lr = 1e-3,
    n_epochs = 30_000,
    lambda_IC = 100.0,
    lambda_BC = 1.0,
    eps_causal = 10.0,
    r3_freq = 500  # Epoch interval for R3 resampling
)



### Initial Condition ($u_0$)

We define two spatial initial profiles $u(0, x, y) \in [-1, 1]$ for the phase-field dynamics:

1. **Elliptical Interface:** Constructs an elliptic domain centered at $(0.5, 0.5)$ with interface transition thickness controlled by $\varepsilon$. The boundary is smoothed using the canonical hyperbolic tangent profile $u_0(x,y) = \tanh\left(\frac{R - d(x,y)}{\sqrt{2}\varepsilon}\right)$.
2. **Alternative — Random Phase Field (Commented):** Generates a smooth, multi-scale stochastic field by superimposing spatial sinusoidal Fourier modes ($k \in [1, 5]$) with randomized phase shifts, mapped into $[-1, 1]$ via $\tanh$.

In [ ]:
# ==============================================================================
# Option 1: Elliptical Interface 
# ==============================================================================
def u_init(point_IC: jax.Array, eps: float = 2.5e-2) -> jax.Array:
    """
    Computes the initial field profile u(0, x, y) representing an elliptical interface.
    
    Args:
        point_IC (jax.Array): Initial point coordinate tensor [t=0, x, y] of shape (3,).
        eps (float): Interface transition layer thickness.
        
    Returns:
        jax.Array: Initial field value u_0 of shape ().
    """
    x, y = point_IC[1], point_IC[2]
    radius = 5.0 * eps
    dists = jnp.sqrt((x - 0.5) ** 2 + 0.49 * (y - 0.5) ** 2)
    u_IC = jnp.tanh((radius - dists) / (jnp.sqrt(2.0) * eps))
    return u_IC

# ==============================================================================
# Option 2: Random Smooth Field (Alternative)
# ==============================================================================
# def u_init(point_IC: jax.Array, seed: int = 42) -> jax.Array:
#     """
#     Computes a smooth random initial field u(0, x, y) via superposition of 
#     low-frequency sinusoidal modes with random phase shifts.
#     
#     Args:
#         point_IC (jax.Array): Initial point coordinate tensor [t=0, x, y] of shape (3,).
#         seed (int): PRNG seed for reproducible random phases.
#         
#     Returns:
#         jax.Array: Initial field value u_0 of shape ().
#     """
#     x, y = point_IC[1], point_IC[2]
# 
#     key = jax.random.PRNGKey(seed)
#     freqs = jnp.arange(1, 6)
#     phases = jax.random.uniform(key, (2, len(freqs)), minval=0.0, maxval=2.0 * jnp.pi)
#     
#     noise = sum(
#         jnp.sin(2.0 * jnp.pi * k * x + phases[0, i]) * 
#         jnp.sin(2.0 * jnp.pi * k * y + phases[1, i]) 
#         for i, k in enumerate(freqs)
#     )
#     u_IC = jnp.tanh(noise / jnp.sqrt(len(freqs)))
#     return u_IC

### Define Residuals and Loss functions

We define the residuals (IC, BC, PDE) at **one point**. Then we define the loss function on the colocation points with the help of `jax.vmap`:

In [ ]:

def res_PDE(model: PINN, point_PDE: jax.Array, eps: float = 2.5e-2) -> jax.Array:
    """
    Evaluates the strong-form Allen-Cahn PDE residual at a single point (t, x, y).
    
    PDE: \partial_t u - \varepsilon^2 (\partial_{xx} u + \partial_{yy} u) - (u - u^3) = 0
    
    Args:
        model (PINN): Current PINN model parameters.
        point_PDE (jax.Array): Point tensor [t, x, y] of shape (3,).
        eps (float): Interface width parameter.
        
    Returns:
        jax.Array: Scalar PDE residual value of shape ().
    """
    u_val = model(point_PDE)
    
    # First-order gradient w.r.t (t, x, y)
    grad_u = jax.grad(model)(point_PDE)
    u_t = grad_u[0]
    
    # Second-order spatial derivatives
    u_xx = jax.grad(lambda pt: jax.grad(model)(pt)[1])(point_PDE)[1]
    u_yy = jax.grad(lambda pt: jax.grad(model)(pt)[2])(point_PDE)[2]

    residual = u_t - ((eps**2) * (u_xx + u_yy) + (u_val - u_val**3))
    return residual


def res_IC(model: PINN, point_IC: jax.Array, eps: float = 2.5e-2) -> jax.Array:
    """
    Evaluates the initial condition residual u(0, x, y) - u_init(0, x, y).
    
    Args:
        model (PINN): Current PINN model parameters.
        point_IC (jax.Array): Initial condition coordinate point [0, x, y] of shape (3,).
        eps (float): Interface parameter.
        
    Returns:
        jax.Array: Scalar IC residual of shape ().
    """
    u_val = model(point_IC)
    u_target = u_init(point_IC, eps=eps)
    return u_val - u_target


# Vectorized residual evaluations across structured domain slices
res_PDE_vec = jax.vmap(
    jax.vmap(res_PDE, in_axes=(None, 0, None)), 
    in_axes=(None, 0, None)
)


# Jit the loss function to accelerate the training process !
@eqx.filter_jit
def non_causal_loss(
    model: PINN,
    points_PDE: jax.Array,
    points_IC: jax.Array,
    points_BC: Tuple[jax.Array, jax.Array, jax.Array, jax.Array],
    lambda_IC: float = 100.0,
    lambda_BC: float = 1.0,
    N_t: int = 40,
    eps: float = 2.5e-2,
) -> Tuple[jax.Array, Dict[str, jax.Array]]:
    """
    Computes the standard non-causal PINN loss with uniform temporal weighting across space-time.

    Args:
        model: The Equinox PINN neural network model.
        points_PDE: Collocation points for PDE residual evaluation (shape: [N_pde, d]).
        points_IC: Initial condition points (shape: [N_ic, d_spatial]).
        points_BC: Tuple of (left, right, top, bottom) boundary points.
        lambda_IC: Loss weight multiplier for the initial condition.
        lambda_BC: Loss weight multiplier for the boundary conditions.
        N_t: Temporal discretization grid size (reserved for temporal binning/causal comparisons).
        eps: Phase-field / regularization parameter passed to IC residual.

    Returns:
        total_loss: Scalar loss used for gradient computation.
        aux: Dictionary containing individual residual components for tracking.
    """

# ============================
# EXERCISE: Define loss_PDE and loss_IC
# ============================
#    loss_PDE =
#
#    loss_IC =
#

    # Periodic Boundary Condition Loss
    points_left, points_right, points_top, points_bottom = points_BC
    loss_bc_x = jnp.mean((jax.vmap(model)(points_left) - jax.vmap(model)(points_right)) ** 2)
    loss_bc_y = jnp.mean((jax.vmap(model)(points_bottom) - jax.vmap(model)(points_top)) ** 2)
    loss_BC = loss_bc_x + loss_bc_y

    # Total combined loss
    total_loss = loss_PDE + lambda_IC * loss_IC + lambda_BC * loss_BC


    return total_loss


### Domain Sampling
Samples training points across space, time, and boundaries:

In [ ]:
def sample_pde_points(key: jax.Array, N_t: int, N_p: int, T0: float, Tf: float) -> jax.Array:
    """
    Generates time-ordered PDE collocation points structured as temporal slices.
    
    Args:
        key (jax.Array): PRNG key.
        N_t (int): Number of discrete temporal slices.
        N_p (int): Number of spatial collocation points per temporal slice.
        T0 (float): Domain start time.
        Tf (float): Domain final time.
        
    Returns:
        jax.Array: Collocation points tensor of shape (N_t, N_p, 3).
    """
    ts = jnp.linspace(T0, Tf, N_t)
    t_grid = jnp.broadcast_to(ts[:, None, None], (N_t, N_p, 1))
    spatial_pts = jax.random.uniform(key, (N_t, N_p, 2))
    return jnp.concatenate([t_grid, spatial_pts], axis=-1)


def sample_initial_and_boundary_points(
    key: jax.Array, n_IC: int, n_BC: int, T0: float, Tf: float
) -> Tuple[jax.Array, Tuple[jax.Array, jax.Array, jax.Array, jax.Array]]:
    """
    Samples initial condition (t=0) and periodic boundary condition coordinates.
    
    Args:
        key (jax.Array): PRNG key.
        n_IC (int): Number of initial condition points.
        n_BC (int): Total number of boundary condition points.
        T0 (float): Initial time.
        Tf (float): Final time.
        
    Returns:
        Tuple[jax.Array, Tuple[jax.Array, jax.Array, jax.Array, jax.Array]]:
            - points_IC: Array of shape (n_IC, 3)
            - points_BC: Tuple containing (left, right, top, bottom) arrays each of shape (n_BC // 2, 3)
    """
    key, ic_key, bc_key1, bc_key2 = jr.split(key, 4)
    
    # 1. Initial Condition Points (LHS spatial sampling)
    lhs_sampler_IC = qmc.LatinHypercube(d=2, seed=42)
    spatial_IC = jnp.array(lhs_sampler_IC.random(n_IC))
    t_init_IC = jnp.zeros((n_IC, 1))
    points_IC = jnp.concatenate([t_init_IC, spatial_IC], axis=-1)

    # 2. Boundary Points (Periodic pairs in X and Y directions)
    n_half = n_BC // 2
    
    # X-direction periodicity (pairing x = 0 and x = 1 at matching y and t)
    t_bc_x = jr.uniform(bc_key1, (n_half, 1), minval=T0, maxval=Tf)
    y_bc_x = jr.uniform(bc_key1, (n_half, 1), minval=0.0, maxval=1.0)
    points_left = jnp.hstack([t_bc_x, jnp.zeros_like(y_bc_x), y_bc_x])
    points_right = jnp.hstack([t_bc_x, jnp.ones_like(y_bc_x), y_bc_x])

    # Y-direction periodicity (pairing y = 0 and y = 1 at matching x and t)
    t_bc_y = jr.uniform(bc_key2, (n_half, 1), minval=T0, maxval=Tf)
    x_bc_y = jr.uniform(bc_key2, (n_half, 1), minval=0.0, maxval=1.0)
    points_bottom = jnp.hstack([t_bc_y, x_bc_y, jnp.zeros_like(x_bc_y)])
    points_top = jnp.hstack([t_bc_y, x_bc_y, jnp.ones_like(x_bc_y)])

    points_BC = (points_left, points_right, points_top, points_bottom)
    return points_IC, points_BC



## Retain–Resample–Release (R3) Sampling in PINNs

Physics-Informed Neural Networks (PINNs) can suffer from **propagation failures**, where the network learns the solution well in certain regions but fails to propagate accuracy across the full spatiotemporal domain. This often happens in time-dependent PDEs, where later time steps inherit and amplify earlier errors.

To address this, the **Retain–Resample–Release (R3) sampling** strategy was introduced in  
**"Mitigating Propagation Failures in Physics-Informed Neural Networks using Retain–Resample–Release (R3) Sampling"**.

### Core Idea
R3 improves sampling efficiency by dynamically managing the collocation points during training:
1. **Retain**: Keep a subset of previously sampled points where the PDE residual is still high, ensuring persistent attention to difficult regions.  
2. **Resample**: Replace some points with newly sampled ones (e.g., via Latin Hypercube or uniform sampling) to maintain exploration of the domain.  
3. **Release**: Remove points that are consistently well satisfied (low residual), freeing capacity for harder regions.  

### Advantages
- Prevents the network from forgetting challenging regions.  
- Mitigates error propagation in time-dependent PDEs.  
- Balances **exploitation** (focus on difficult points) and **exploration** (cover the full domain).  
- Provides a more stable and accurate training process compared to static or naive resampling.  

### Reference
- [Mitigating Propagation Failures in Physics-Informed Neural Networks using Retain–Resample–Release (R3) Sampling](https://arxiv.org/abs/2207.02338)  

In [ ]:
def r3_sampling(
    model: PINN, key: jax.Array, points_PDE: jax.Array, points_IC: jax.Array, eps: float = 2.5e-2
) -> Tuple[jax.Array, jax.Array]:
    """
    Performs Retain-Resample-Release (R3) dynamic collocation point updates.
    
    Args:
        model (PINN): Current PINN parameter state.
        key (jax.Array): PRNG key for sub-sampling.
        points_PDE (jax.Array): Current PDE points tensor of shape (N_t, N_p, 3).
        points_IC (jax.Array): Current IC points tensor of shape (n_IC, 3).
        eps (float): Physics thickness scale.
        
    Returns:
        Tuple[jax.Array, jax.Array]: Updated (points_PDE, points_IC) tensors matching original shapes.
    """
    key, pde_key, ic_key = jr.split(key, 3)
    N_t, N_p, _ = points_PDE.shape

    # 1. Resample PDE interior points per time slice
    r_PDE = res_PDE_vec(model, points_PDE, eps)  # Shape: (N_t, N_p)
    time_keys = jr.split(pde_key, N_t)
    resampled_PDE_slices = []

    for i in range(N_t):
        pts_i = points_PDE[i]
        r_i = r_PDE[i]
        t_val = pts_i[0, 0]
        
        threshold_i = jnp.abs(r_i).mean()
        retained_pts = pts_i[jnp.abs(r_i) >= threshold_i]
        
        n_retained = retained_pts.shape[0]
        n_new = N_p - n_retained
        
        new_spatial = jr.uniform(time_keys[i], (n_new, 2))
        new_t = jnp.full((n_new, 1), t_val)
        new_pts = jnp.hstack([new_t, new_spatial])
        
        resample_slice = jnp.vstack([retained_pts, new_pts])
        resampled_PDE_slices.append(resample_slice)
        
    resampling_PDE = jnp.stack(resampled_PDE_slices, axis=0)

    # 2. Resample Initial Condition (IC) points
    n_IC = points_IC.shape[0]
    r_IC = jax.vmap(res_IC, in_axes=(None, 0, None))(model, points_IC, eps)
    threshold_IC = jnp.abs(r_IC).mean() 
    
    r3_points_IC = points_IC[jnp.abs(r_IC) >= threshold_IC]
    n_r3IC = r3_points_IC.shape[0]
    
# ============================
# EXERCISE: Complet the rest
# ============================
#    resampling_IC =  jnp.vstack( [r3_points_IC, new_points_IC])

    return resampling_PDE, resampling_IC


### Training Pipeline

Define the step function and main optimization loop:

In [ ]:
def make_step(loss_fn: Callable):
    """Creates a JIT-compiled step function bound to a specific loss_fn."""
    
    @eqx.filter_jit
    def step(
        model: PINN,
        opt_state: optax.OptState,
        optimizer: optax.GradientTransformation,
        points_PDE: jax.Array,
        points_IC: jax.Array,
        points_BC: Tuple[jax.Array, jax.Array, jax.Array, jax.Array],
        eps_causal: float,
        lambda_IC: float,
        lambda_BC: float,
        N_t: int,
        eps: float,
    ) -> Tuple[PINN, optax.OptState, jax.Array]:
        
        # loss_fn is cleanly captured in the outer closure
        loss_fonction = partial(
            loss_fn,
            lambda_IC=lambda_IC,
            lambda_BC=lambda_BC,
            N_t=N_t,
            eps=eps
        )
        total_loss, grads = eqx.filter_value_and_grad(loss_fonction)(model, points_PDE, points_IC, points_BC)
        updates, opt_state = optimizer.update(grads, opt_state, model)
        model = eqx.apply_updates(model, updates)
        return model, opt_state, total_loss

    return step


def train_pinn(
    config: Config,
    key: jax.Array,
    loss_fn: Callable
) -> Tuple[PINN, List[float], jax.Array, jax.Array]:
    """
    Full training pipeline incorporating network initialization, causality loss, and R3 resampling.
    
    Args:
        config (Config): NamedTuple containing domain and training hyperparameters.
        key (jax.Array): Master PRNG seed.
        
    Returns:
        Tuple[PINN, List[float], jax.Array, jax.Array]:
            - best_model: Serialized best PINN model evaluated over training history.
            - loss_history: List of float scalar loss values per epoch.
            - points_PDE: Final adapted PDE sampling tensor (N_t, N_p, 3).
            - points_IC: Final adapted IC sampling tensor (n_IC, 3).
    """
    key, model_key, pde_key, ic_bc_key = jr.split(key, 4)
    
    # Initialize Model & Optimizer
    model = PINN(in_size=3, out_size="scalar", width_size=100, depth=4, key=model_key)
    optimizer = optax.adam(config.lr)
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

    # Initialize Sampling Domains
    points_PDE = sample_pde_points(pde_key, config.N_t, config.n_PDE, config.T0, config.Tf)
    points_IC, points_BC = sample_initial_and_boundary_points(
        ic_bc_key, config.n_IC, config.n_BC, config.T0, config.Tf
    )

    step = make_step(loss_fn)

    loss_history: List[float] = []
    best_loss = 1e9
    best_model = model

    pbar = trange(config.n_epochs, desc="Training PINN")
    for epoch in pbar:
# ============================
# EXERCISE: Complete the traning loop
# ============================
        
        loss_val = float(total_loss)
        loss_history.append(loss_val)

        if loss_val < best_loss:
            best_loss = loss_val
            best_model = model

        # Dynamic R3 Resampling Step
        if (epoch + 1) % config.r3_freq == 0:
    # ============================
    # EXERCISE: Apply the r3 sampling to update colocation points
    # ============================

        if epoch % 50 == 0 or epoch == config.n_epochs - 1:
            pbar.set_postfix({"Loss": f"{loss_val:.4e}", "Best": f"{best_loss:.4e}"})

    return best_model, loss_history, points_PDE, points_IC



Now, it's time to initialize the model and kicks off training!

In [ ]:
key = jr.PRNGKey(42)

# Cache initial sampling states for visualization comparison
key_init, key_train = jr.split(key)
pde_init_key, ic_init_key = jr.split(key_init)
points_PDE_init = sample_pde_points(pde_init_key, config.N_t, config.n_PDE, config.T0, config.Tf)
points_IC_init, points_BC_init = sample_initial_and_boundary_points(
    ic_init_key, config.n_IC, config.n_BC, config.T0, config.Tf
)

# Train Model
print("\n=== Launching PINN Training for 2D Allen-Cahn Equation ===")
trained_model, loss_hist, points_PDE_final, points_IC_final = train_pinn(config, key_train, loss_fn=non_causal_loss)

# Save model checkpoint
serialized_filename = "pinns_2d_allen-cahn.eqx"
eqx.tree_serialise_leaves(serialized_filename, trained_model)
print(f"\nModel successfully serialized and saved to '{serialized_filename}'.")

# Render Visualization Outputs
print("\n=== Generating Visualizations ===")
plot_loss_history(loss_hist)
plot_sampling_points(points_IC_init, points_IC_final, title="Initial Condition Points (IC)")
plot_sampling_points(points_PDE_init, points_PDE_final, title="Interior PDE Points")
plot_solution_slices(trained_model, config, num_slices=9)

### Reference Solution via Implicit Spectral Scheme

To obtain reference trajectories for validation, we compute the ground-truth dynamics using an **implicit operator-splitting Fourier spectral solver**.



In [ ]:
from utils.helper import splitting_next
spectral_solver = eqx.filter_jit(partial(splitting_next, N_x=config.N_x, N_y=config.N_y, dx=1/config.N_x, eps=config.eps))
xs, ys = jnp.meshgrid(
    jnp.linspace(config.x_lb, config.x_rb, config.N_x),
    jnp.linspace(config.y_lb, config.y_ub, config.N_y),
)
ts = jnp.zeros_like(xs)
reg_grid = jnp.stack([ts, xs, ys], axis=-1)
U0 =jax.vmap(jax.vmap(u_init))(reg_grid)
U = U0 
dt = (config.Tf-config.T0)/config.N_t
step = config.N_t//8
plt.figure(figsize=(12, 12))
plt.subplot(3, 3, 1)
plt.imshow(U, extent=[config.x_lb, config.x_rb, config.y_lb, config.y_ub], cmap='jet',vmin=-1,vmax=1)
plt.colorbar()
plt.title("Initial Condition")
for i in range(8):
    plt.subplot(3, 3, i+2)
    for _ in range(step):
        U = spectral_solver(U, dt=dt)
    t_i = step*dt*(i+1)
    plt.title(f"T={t_i:.1f}")
    plt.imshow(U, extent=[config.x_lb, config.x_rb, config.y_lb, config.y_ub], cmap='jet', vmin=-1, vmax=1)
    plt.colorbar()



### Solution Analysis: PINN vs. Reference

What can we observe when comparing the PINN prediction against the reference spectral solution?

* **Accuracy & Interface Tracking:** How well does the PINN preserve the interface dynamics and phase field profile over time?
* **Spatial Error Distribution:** Where are the largest pointwise errors concentrated (e.g., along sharp transition layers vs. smooth interior regions)?
* **Temporal Accumulation:** Does the prediction quality remain stable or degrade at later time steps ($T \to T_f$)?

# (Advanced) Causal Training Strategy for PINNs

Standard PINN training minimizes the PDE residual uniformly across the entire spatio-temporal domain $[0, T]\times [0,1]^2$. However, evolutionary physical systems strictly adhere to **causality**: the solution state at time $t$ depends fundamentally on the solution at preceding times $t' < t$.

In complex or chaotic dynamics, minimizing errors at late times before resolving early-time features causes optimization stagnation in non-physical local minima.

---

### Mathematical Formulation

To enforce temporal causality (Wang et al., 2022), the time domain $[0, T]$ is discretized into $N_t$ sequential intervals $\{t_m\}_{m=1}^{N_t}$. We assign a dynamic causal weight $w_m \in (0, 1]$ to the PDE residual loss at each time step $t_m$:

$$\mathcal{L}_{\text{causal}}(\theta) = \frac{1}{N_t} \sum_{m=1}^{N_t} w_m \mathcal{L}_m(\theta) + \lambda_{\text{IC}} \mathcal{L}_{\text{IC}}(\theta) + \lambda_{\text{BC}} \mathcal{L}_{\text{BC}}(\theta)$$

where $\mathcal{L}_m(\theta)$ denotes the mean PDE residual at time bin $t_m$. 

The weights $w_m$ are updated dynamically based on the cumulative residual of preceding time steps:

$$w_1 = 1, \qquad w_m = \exp\left( -\epsilon \sum_{k=1}^{m-1} \mathcal{L}_k(\theta) \right) \quad \text{for } m = 2, \dots, N_t$$

---

### Key Hyperparameters & Mechanics

* **Causality Steepness ($\epsilon$):** 
  * $\epsilon = 0 \implies w_m = 1, \forall m$, recovering standard non-causal training.
  * $\epsilon > 0$ forces the model to focus strictly on resolving early time intervals before propagating optimization to later times.
* **Stop-Gradient Operation:** The calculation of $w_m$ must use `jax.lax.stop_gradient` to treat the loss weights as non-differentiable multipliers during gradient descent.

In [ ]:
@eqx.filter_jit
def residus_and_weights(
    model: PINN, points_PDE: jax.Array, eps_causal: float = 100.0, N_t: int = 40, eps: float = 2.5e-2
) -> Tuple[jax.Array, jax.Array]:
    """
    Computes time-slice mean squared PDE residuals and causality weighting coefficients.
    
    Args:
        model (PINN): Model neural network.
        points_PDE (jax.Array): Tensor of shape (N_t, N_p, 3).
        eps_causal (float): Causality exponential damping parameter.
        N_t (int): Total number of time slices.
        eps (float): Interface width parameter.
        
    Returns:
        Tuple[jax.Array, jax.Array]:
            - L_t: Mean squared residuals per time slice of shape (N_t,)
            - W: Causality weights per time slice of shape (N_t,)
    """
    M = jnp.tril(jnp.ones((N_t, N_t)), k=-1) # M is a matrice with subdiagonal equal to 1 and 0 otherwise
    r_PDE = res_PDE_vec(model, points_PDE, eps)  # Shape: (N_t, N_p)
    # ============================
    # EXERCISE: Define the L_t and W with help of M
    # ============================
    return L_t, W

@eqx.filter_jit
def causality_loss(
    model: PINN,
    points_PDE: jax.Array,
    points_IC: jax.Array,
    points_BC: Tuple[jax.Array, jax.Array, jax.Array, jax.Array],
    eps_causal: float = 100.0,
    lambda_IC: float = 100.0,
    lambda_BC: float = 1.0,
    N_t: float = 32,
    eps: float = 2.5e-2
) -> jax.Array:
    """
    Computes causality-weighted total PINN loss (L_causal_PDE + lambda_IC * L_IC + lambda_BC * L_BC).
    
    Args:
        model (PINN): Current PINN model.
        points_PDE (jax.Array): PDE collocation points of shape (N_t, N_p, 3).
        points_IC (jax.Array): Initial condition points of shape (n_IC, 3).
        points_BC (Tuple[jax.Array, ...]): Tuple of boundary condition slices.
        eps_causal (float): Causality hyperparameter.
        lambda_IC (float): Initial condition loss weight coefficient.
        lambda_BC (float): Boundary condition loss weight coefficient.
        N_t (int): Number of temporal slices.
        eps (float): Interface width parameter.
        
    Returns:
        jax.Array: Scalar total loss value of shape ().
    """
    # Initial Condition Loss
    r_IC = jax.vmap(res_IC, in_axes=(None, 0, None))(model, points_IC, eps)
    loss_IC = jnp.mean(r_IC**2)

    # Periodic Boundary Condition Loss
    points_left, points_right, points_top, points_bottom = points_BC
    loss_bc_x = jnp.mean((jax.vmap(model)(points_left) - jax.vmap(model)(points_right)) ** 2)
    loss_bc_y = jnp.mean((jax.vmap(model)(points_bottom) - jax.vmap(model)(points_top)) ** 2)
    loss_BC = loss_bc_x + loss_bc_y

    # Temporal Causal PDE Loss
    # ============================
    # EXERCISE: Compute the Causal loss_PDE 
    # ============================

    return loss_PDE + lambda_IC * loss_IC + lambda_BC * loss_BC

### Causal Training Execution
Initializes the model and kicks off training with the causality loss!

In [ ]:
key = jr.PRNGKey(42)

# Cache initial sampling states for visualization comparison
key_init, key_train = jr.split(key)
pde_init_key, ic_init_key = jr.split(key_init)
points_PDE_init = sample_pde_points(pde_init_key, config.N_t, config.n_PDE, config.T0, config.Tf)
points_IC_init, points_BC_init = sample_initial_and_boundary_points(
    ic_init_key, config.n_IC, config.n_BC, config.T0, config.Tf
)

# Train Model
print("\n=== Launching PINN Training for 2D Allen-Cahn Equation ===")
trained_model, loss_hist, points_PDE_final, points_IC_final = train_pinn(
    config=config, key=key_train, loss_fn=causality_loss
)

# Save model checkpoint
serialized_filename = "pinns_2d_allen-cahn-with_causal.eqx"
eqx.tree_serialise_leaves(serialized_filename, trained_model)
print(f"\nModel successfully serialized and saved to '{serialized_filename}'.")

# Render Visualization Outputs
print("\n=== Generating Visualizations ===")
plot_loss_history(loss_hist)
plot_sampling_points(points_IC_init, points_IC_final, title="Initial Condition Points (IC)")
plot_sampling_points(points_PDE_init, points_PDE_final, title="Interior PDE Points")
plot_solution_slices(trained_model, config, num_slices=9)

### Causality Training: Analysis & Improvements

How did the temporal causality mechanism perform, and how can it be pushed further?

Remark: In practice, we increase gradually the tolerance parameter `eps_causal` during training, which corresponds to [Curriculum Learning](https://en.wikipedia.org/wiki/Curriculum_learning).

# (Optional) Fourier Neural Operator (FNO) & Physics-Informed Extensions

The Fourier Neural Operator (**FNO**) [[1]](#1) learns mappings between infinite-dimensional function spaces to solve parametric PDEs directly as operator maps $G: a \mapsto u$. Extending FNO with physics-informed constraints gives the Physics-Informed Neural Operator (**PINO**) [[2]](#2), enabling data-free training and fine-tuning via exact PDE residual minimization.

---

### Core Architecture

For an input field $a(x)$, the architecture computes $a \mapsto u$ in three stages:

1. **Lifting ($P$):** Projects input to hidden channels: $v_0(x) = P(a(x), x)$.
2. **Fourier Layers ($L$ steps):** Updates features using global spectral convolutions:
   $$v_{l+1}(x) = \sigma \left( W_l v_l(x) + \mathcal{F}^{-1} \left( R_l \cdot (\mathcal{F} v_l) \right)(x) \right)$$
   * $\mathcal{F}, \mathcal{F}^{-1}$ denote the FFT and inverse FFT.
   * $R_l$ parameterizes a truncated set of lower Fourier modes $k \le k_{\max}$.
   * $W_l$ is a local residual connection.
3. **Projection ($Q$):** Maps hidden features back to physical output space: $u(x) = Q(v_L(x))$.

<img src="FNO-illustration.png" width="800">
---

### Key Advantages

* **Discretization Invariance:** Spectral weights $R_l$ are continuous; models trained on coarse grids evaluate on fine grids without retraining.
* **Global Receptive Field:** FFT-based convolutions capture domain-wide interactions in $\mathcal{O}(N \log N)$ time per layer.
* **Physics-Informed Regularization (PINO):** Evaluates exact operator residuals on arbitrary grids via spectral/automatic differentiation, eliminating the need for ground-truth simulation data.

---

### References

<a id="1">[1]</a> Li, Z. et al. (2020). *Fourier Neural Operator for Parametric Partial Differential Equations*. ICLR 2021.  
<a id="2">[2]</a> Li, Z. et al. (2021). *Physics-Informed Neural Operator for Learning Partial Differential Equations*. arXiv:2111.03794.

---

*For complete implementation details and spectral layer definitions, refer to `utils/FNO.py`.*

[Iterative Fourier Neural Operator](https://openreview.net/forum?id=4lAS8tI23u&referrer=%5Bthe%20profile%20of%20Xiaotian%20Liu%5D(%2Fprofile%3Fid%3D~Xiaotian_Liu2)) has recently been very successful in numerous application. Build it by yourself with the help of the spectral layers defined in `utils/FNO.py`.